# Module 3: Synthetic Corpus Generation & Validation

Generate word-level signals by concatenating letter recordings with boundary noise injection.

**Process:**
1. Load dataset and word list
2. For each word: sample random letter recordings, concatenate, inject boundary noise
3. Store (noisy_signal, clean_signal, label) tuples in HDF5
4. Visualize: confirm noise patterns match spec

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import json
import yaml
import h5py

sys.path.insert(0, os.path.abspath('..'))

from models.dataset import get_dataset
from models.synthesis import synthesize_corpus

# Load dataset and wordlist
dataset = get_dataset("../data/icub_braille_raw")

with open('../data/wordlist_splits.json', 'r') as f:
    splits = json.load(f)

wordlist = sorted(set(splits['train'] + splits['val'] + splits['test']))
print(f"✓ Loaded {len(wordlist)} words")
print(f"✓ Dataset ready with {len(dataset.get_all_letters('a'))} recordings per character (sample)")

## Generate Synthetic Corpus

Create the full corpus with boundary noise injection.

In [ ]:
# Generate corpus
corpus_path = synthesize_corpus(
    dataset,
    wordlist,
    splits,
    noise_config_path='../config/noise.yaml',
    output_path='../data/corpus.h5',
    samples_per_word={'train': 5, 'val': 2, 'test': 2}
)

print(f"✓ Corpus saved to {corpus_path}")

## Acceptance Check: Boundary Noise Visualization

Plot samples showing noisy vs clean signals with boundary regions highlighted.

In [ ]:
# Load and visualize random samples
with h5py.File('../data/corpus.h5', 'r') as f:
    val_group = f['val']
    sample_ids = sorted(list(val_group.keys()))[:5]
    
    fig, axes = plt.subplots(len(sample_ids), 2, figsize=(14, 3*len(sample_ids)))
    if len(sample_ids) == 1:
        axes = [axes]
    
    for row, sample_id in enumerate(sample_ids):
        sample = val_group[sample_id]
        noisy = sample['noisy'][:]  # (T, 12)
        clean = sample['clean'][:]
        boundaries = sample['boundaries'][:]
        label = sample.attrs['label']
        
        # Plot channel 0
        ax = axes[row][0]
        ax.plot(clean[:, 0], label='clean', linewidth=1, alpha=0.7)
        ax.plot(noisy[:, 0], label='noisy', linewidth=1, alpha=0.7)
        
        # Highlight boundaries
        for b in boundaries[1:]:
            ax.axvline(b, color='red', linestyle='--', alpha=0.5, linewidth=0.5)
        
        ax.set_title(f'Word: "{label}" (Sample {sample_id})')
        ax.legend()
        ax.set_ylabel('Channel 0')
        
        # Plot noisy - clean diff
        ax = axes[row][1]
        diff = np.abs(noisy[:, 0] - clean[:, 0])
        ax.plot(diff, linewidth=1, color='orange')
        
        for b in boundaries[1:]:
            ax.axvline(b, color='red', linestyle='--', alpha=0.5, linewidth=0.5)
        
        ax.set_title(f'Noise Magnitude')
        ax.set_ylabel('|noisy - clean|')
    
    axes[-1][0].set_xlabel('Timesteps')
    axes[-1][1].set_xlabel('Timesteps')
    
    plt.tight_layout()
    plt.savefig('../data/corpus_validation.png', dpi=150)
    plt.show()
    
    print("✓ Visualization saved to data/corpus_validation.png")
    print("✓ Visual confirmation: boundaries show increased noise (smeared transitions)")